In [1]:
import numpy as np
import pandas as pd

trans_df = pd.read_csv("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Small_Trans.csv").sample(n=500000)
trans_df.head(5)

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
1808191,2022/09/02 06:00,20,802EABEB0,220270,80E25DFF0,9661.410000,Euro,9661.410000,Euro,Wire,0
2964711,2022/09/04 09:48,169309,819863821,214,819863911,2.615626,Bitcoin,2.615626,Bitcoin,Bitcoin,0
1611079,2022/09/02 01:35,117301,809589750,18840,80A648280,115.790000,US Dollar,115.790000,US Dollar,Cheque,0
6884478,2022/09/10 20:07,41014,810DFA120,19,8110BC990,2508.650000,Mexican Peso,2508.650000,Mexican Peso,Cheque,0
6089511,2022/09/09 08:53,15168,8028CE9E0,692,8039B55A0,341.350000,Yuan,341.350000,Yuan,ACH,0


In [2]:
# Analyze timestamps.
print(f"Timestamp range: [{trans_df["Timestamp"].min()},{trans_df["Timestamp"].max()}]")

Timestamp range: [2022/09/01 00:00,2022/09/17 02:32]


In [3]:
# Analyze transfers. Check for duplicate Account Numbers in different banks.
df_senders = trans_df[['From Bank', 'Account']].rename(columns={
    'From Bank': 'Bank', 
})
df_receivers = trans_df[['To Bank', 'Account.1']].rename(columns={
    'To Bank': 'Bank', 
    'Account.1': 'Account'
})
df_bank_accounts = pd.concat([df_senders, df_receivers],ignore_index=True)
df_bank_counts = df_bank_accounts.drop_duplicates().groupby('Account')['Bank'].count()
df_bank_counts[df_bank_counts > 1]

Series([], Name: Bank, dtype: int64)

In [4]:
#Filter non USD transactions.
trans_usd_df = trans_df[trans_df['Payment Currency'] == "US Dollar"]
print("SIZE:", trans_usd_df.shape[0])

SIZE: 184033


In [5]:
# Analyze accounts.
accounts_df = pd.read_csv("/kaggle/input/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml/LI-Small_accounts.csv")
print("SIZE:", accounts_df.shape[0])
accounts_df.head(5)

SIZE: 712688


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930


In [6]:
trans_usd_sept_1st_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/01') & (trans_usd_df["Timestamp"] <= '2022/09/06')]
print("SIZE:", trans_usd_sept_1st_df.shape[0])

SIZE: 100497


In [7]:
def filter_function(x):
    unique_account_size = x.groupby(["To Bank", "Account.1"]).size().size
    return  unique_account_size > 5 and unique_account_size < 10
ranged_trans_usd_sept_df = trans_usd_sept_1st_df.groupby(["From Bank", "Account"]).filter(filter_function)
print("SIZE:", ranged_trans_usd_sept_df.shape[0])

SIZE: 1459


In [8]:
#1. Amount, source and target accounts for transactions of less than 50 USD.

low_profile_transactions = trans_usd_df[trans_usd_df['Amount Paid'] < 50]
low_profile_transactions = low_profile_transactions[['From Bank', 'Account', 'To Bank','Account.1', 'Amount Paid']]
low_profile_transactions

,From Bank,Account,To Bank,Account.1,Amount Paid
1458228,113213,8057E4A20,20,8076362A0,26.55
147676,224231,818AD4D20,224231,818AD4D20,7.65
1845561,1291,805666C50,116703,815A820F0,4.93
114291,237102,8136E3F30,237102,8136E3F30,20.57
3130989,252222,81BBA3DC0,223057,812A452A0,7.18
...,...,...,...,...,...
2839293,2860,808049BF0,5394,812D36CF0,4.27
158563,58750,81AC0EAB0,58750,81AC0EAB0,5.39
1228293,21939,805614020,1638,805CC7060,17.30
479600,21828,81BDFF340,21828,81BDFF340,4.37


In [9]:
#2. Max amount by source bank, source Bank Id and Bank Name considering all the transactions.

max_amount_trans_usd_idx = trans_usd_df.groupby(["From Bank"])["Amount Paid"].idxmax()
max_amount_trans_usd = trans_usd_df.loc[max_amount_trans_usd_idx]
max_amount_bank = max_amount_trans_usd.merge(accounts_df, left_on="From Bank", right_on="Bank ID")
max_amount_bank[["From Bank", "Account", "Bank Name","Amount Paid"]]

,From Bank,Account,Bank Name,Amount Paid
0,0,8022D5830,Italy Bank #18,50351.13
1,0,8022D5830,Italy Bank #18,50351.13
2,0,8022D5830,Italy Bank #18,50351.13
3,0,8022D5830,Italy Bank #18,50351.13
4,0,8022D5830,Italy Bank #18,50351.13
...,...,...,...,...
411131,376609,81BFDDA00,National Bank of Pittsburgh,440.33
411132,376653,81C030DC0,Bank of Portland,5595.73
411133,376837,81C14E550,Willows Bancorp,209.97
411134,376856,81C16F0A0,Sunrise Bancorp,1560.63


In [10]:
#3. Source account, payment format, and amount of transactions in period [2022-09-06, 2022-11-06] with amount lower than AVG/100 of period [2022-09-01, 2022-09-05] for the same type of transaction.

avg_amounts_per_type = trans_usd_sept_1st_df.groupby(["Payment Format"])["Amount Paid"].mean().reset_index()
trans_usd_sept_2nd_df = trans_usd_df[(trans_usd_df["Timestamp"] >= '2022/09/06') & (trans_usd_df["Timestamp"] <= '2022/09/15')]
trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_df.merge(avg_amounts_per_type, left_on=["Payment Format"], right_on=["Payment Format"]).rename(columns={
    "Amount Paid_x": "Amount Paid",
    "Amount Paid_y": "AVG",
})
lower_trans_usd_sept_2nd_with_avg_df = trans_usd_sept_2nd_with_avg_df[trans_usd_sept_2nd_with_avg_df["Amount Paid"] < trans_usd_sept_2nd_with_avg_df["AVG"] * 0.01]
lower_trans_usd_sept_2nd_with_avg_df[["From Bank", "Account", "Payment Format", "Amount Paid"]]

,From Bank,Account,Payment Format,Amount Paid
0,3500,8045A7FA0,Cheque,384.01
1,70,10042B660,Cheque,548.66
2,2860,802E48330,ACH,1346.67
3,9715,806DB9D10,Cheque,1327.99
5,11701,80DC06030,Cash,356.85
...,...,...,...,...
83527,27009,80BCC1580,Cheque,4563.42
83529,116186,80A8464A0,Wire,545.36
83531,23760,803904110,Cheque,1269.70
83532,17180,80661EE30,Cheque,4.90


In [11]:
#4. Accounts that match the scatter-gather pattern and where the source account has transferred to more than 5 distinct accounts.

accounts_df = ranged_trans_usd_sept_df[["From Bank", "Account", "To Bank", "Account.1"]]
account_pairs_df = accounts_df.merge(accounts_df, left_on=["To Bank", "Account.1"], right_on=["From Bank", "Account"]).rename(columns={
    "From Bank_x": "From Bank",
    "Account_x": "From Account",
    "To Bank_y": "To Bank",
    "Account.1_y": "To Account"
})
account_pairs_df = account_pairs_df[(account_pairs_df["From Bank"] != account_pairs_df["To Bank"]) | (account_pairs_df["From Account"] != account_pairs_df["To Account"])]
account_pairs_df = account_pairs_df.groupby(["From Bank", "From Account", "To Bank", "To Account"], as_index=False).size()
account_pairs_df = account_pairs_df[(account_pairs_df["size"] > 5)]


from_account_pairs_df = account_pairs_df[["From Bank", "From Account"]].rename(columns={
    "From Bank": "Bank",
    "From Account": "Account"
})
to_account_pairs_df = account_pairs_df[["To Bank", "To Account"]].rename(columns={
    "To Bank": "Bank",
    "To Account": "Account"
})
unique_accounts = pd.concat([from_account_pairs_df, to_account_pairs_df]).drop_duplicates()
unique_accounts

,Bank,Account
79,11,800135A60
102,12,800056090
157,12,800205D10
223,20,80010B000
249,20,8002CBEA0
367,1291,8001755E0
79,11,80014CC70
81,1217,800200300
102,224,8000FBE20
103,224,8001DC0F0


In [12]:
#5. Count of transactions of period [2022-09-01, 2022-09-05] with type Wire or ACH, having converted amount for that day less than USD 1.
#Map everything to US Dollars by a fixed table.
TO_US_DOLLARS = {
    "Australian Dollar": 0.72,
    "Bitcoin": 78.33,
    "Brazil Real": 0.20,
    "Canadian Dollar": 0.73,
    "Euro": 1.17,
    "Mexican Peso": 0.06,
    "Ruble": 0.01,
    "Rupee": 0.01,
    "Saudi Riyal":0.27,
    "Shekel": 0.33,
    "Swiss Franc": 1.27,
    "UK Pound":  1.35,
    "US Dollar": 1,
    "Yen": 0.006,
    "Yuan": 0.15
}
trans_sept_1st_df = trans_df[(trans_df["Timestamp"] >= '2022/09/01') & (trans_df["Timestamp"] <= '2022/09/06')]
trans_sept_1st_wire_or_ach_df = trans_sept_1st_df[(trans_sept_1st_df["Payment Format"] == "Wire") | (trans_sept_1st_df["Payment Format"] == "ACH")]
trans_sept_1st_wire_or_ach_converted_df = trans_sept_1st_wire_or_ach_df.copy()
trans_sept_1st_wire_or_ach_converted_df['Amount'] = trans_sept_1st_wire_or_ach_converted_df['Amount Paid'] * trans_sept_1st_wire_or_ach_converted_df['Payment Currency'].map(TO_US_DOLLARS)
print("SIZE:", trans_sept_1st_wire_or_ach_converted_df.shape[0])

SIZE: 37504
